In [4]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from scipy.ndimage import gaussian_filter
from pathlib import Path

# Output
out_path = Path("./data_set/orthomosaic/example_clearcut_orthomosaic.tif")
# Image size (moderate, fast)
width = 3000    # px
height = 2000   # px
bands = 3

# Base random field
rng = np.random.default_rng(42)
base = rng.random((height, width))

# Smooth to create terrain-like texture
soil = gaussian_filter(base, sigma=20)
slash = gaussian_filter(rng.random((height, width)), sigma=6)
veg = gaussian_filter(rng.random((height, width)), sigma=10)

# Normalize
def norm(x):
    return (x - x.min()) / (x.max() - x.min())

soil = norm(soil)
slash = norm(slash)
veg = norm(veg)

# Create clear-cut patterns
clearcut_mask = soil > 0.45
trail_mask = np.zeros_like(clearcut_mask)

# Add linear skid trails
for i in range(6):
    y = rng.integers(0, height)
    trail_mask[y:y+20, :] = 1

trail_mask = gaussian_filter(trail_mask.astype(float), sigma=3) > 0.3

# RGB composition
R = 120 + 90 * soil          # exposed soil
G = 80  + 100 * veg * 0.4    # limited vegetation
B = 60  + 70 * slash * 0.3   # woody debris

# Trails are compacted & darker
R[trail_mask] *= 0.6
G[trail_mask] *= 0.6
B[trail_mask] *= 0.6

# Stack
image = np.stack([R, G, B], axis=0)
image = np.clip(image, 0, 255).astype(np.uint8)

# Georeferencing (example UTM)
pixel_size = 0.05  # 5 cm
transform = from_origin(
    west=500_000,
    north=5_450_000,
    xsize=pixel_size,
    ysize=pixel_size
)

crs = "EPSG:32610"  # UTM Zone 10N

# Write GeoTIFF
with rasterio.open(
    out_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=3,
    dtype="uint8",
    crs=crs,
    transform=transform,
    tiled=True,
    compress="deflate",
    bigtiff="if_safer"
) as dst:
    dst.write(image)

out_path


PosixPath('data_set/orthomosaic/example_clearcut_orthomosaic.tif')

In [1]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from pathlib import Path

# Output path
out_path = Path("./data_set/orthomosaic/example_orthomosaic.tif")

# Create a fake RGB orthomosaic
width = 2000    # pixels
height = 1500   # pixels
bands = 3

# Fake data (smooth gradient + noise, like aerial imagery)
rng = np.random.default_rng(42)
image = np.zeros((bands, height, width), dtype=np.uint8)

x = np.linspace(0, 255, width, dtype=np.uint8)
y = np.linspace(0, 255, height, dtype=np.uint8)

image[0] = np.tile(x, (height, 1))                 # Red gradient
image[1] = np.tile(y[:, None], (1, width))         # Green gradient
image[2] = rng.integers(0, 80, size=(height, width), dtype=np.uint8)  # Blue noise

# Fake georeferencing (UTM-like)
pixel_size = 0.05  # 5 cm / pixel
transform = from_origin(
    west=500_000,     # fake easting
    north=5_450_000,  # fake northing
    xsize=pixel_size,
    ysize=pixel_size
)

crs = "EPSG:32610"  # UTM Zone 10N (example)

# Write GeoTIFF
with rasterio.open(
    out_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=bands,
    dtype=image.dtype,
    crs=crs,
    transform=transform,
    tiled=True,
    compress="deflate"
) as dst:
    dst.write(image)

out_path


PosixPath('example_orthomosaic.tif')